In [1]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [2]:
class TrackedObject:
  def __init__(self, **kwargs):
    super().__setattr__('history', [])
    for key, value in kwargs.items():
        self.__setattr__(key, value)

  def __setattr__(self, name, value):
    size_before = len(self.__dict__)
    super().__setattr__(name, value)
    size_after = len(self.__dict__)
    self.history.append(f"SET {name}={value}, size: {size_before} -> {size_after}")

  def __delattr__(self, name):
    size_before = len(self.__dict__)
    super().__delattr__(name)
    size_after = len(self.__dict__)
    self.history.append(f"DEL {name}, size: {size_before} -> {size_after}")

obj = TrackedObject(x=1, y=2)
obj.z = 3
obj.__str__ = lambda s: "patched"
del obj.x

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': ['SET x=1, size: 1 -> 2', 'SET y=2, size: 2 -> 3', 'SET z=3, size: 3 -> 4', 'SET __str__=<function <lambda> at 0x11221c880>, size: 4 -> 5', 'DEL x, size: 5 -> 4'], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x11221c880>}
SET x=1, size: 1 -> 2
SET y=2, size: 2 -> 3
SET z=3, size: 3 -> 4
SET __str__=<function <lambda> at 0x11221c880>, size: 4 -> 5
DEL x, size: 5 -> 4


## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [3]:
def merge(seqs):
    seqs = [s for s in seqs if s]
    if not seqs:
        return []
    for seq in seqs:
        candidate = seq[0]
        if all(candidate not in s[1:] for s in seqs):
            return [candidate] + merge([s if s[0] != candidate else s[1:] for s in seqs])
    raise TypeError("Cannot create consistent MRO")

def c3_linearize(cls):
    if cls is object:
        return [object]
    bases_mro = [c3_linearize(base) for base in cls.__bases__]
    return [cls] + merge(bases_mro + [list(cls.__bases__)])

class A:
    def method(self): return "A"

class B(A):
    def method(self): return "B"

class C(A):
    def method(self): return "C"

class D(B, C):
    pass

class E(A):
    def method(self): return "E"

class F(D, E):
    pass

print("D MRO (computed):", [c.__name__ for c in c3_linearize(D)])
print("D.__mro__:       ", [c.__name__ for c in D.__mro__])
print()
print("F MRO (computed):", [c.__name__ for c in c3_linearize(F)])
print("F.__mro__:       ", [c.__name__ for c in F.__mro__])

D MRO (computed): ['D', 'B', 'C', 'A', 'object']
D.__mro__:        ['D', 'B', 'C', 'A', 'object']

F MRO (computed): ['F', 'D', 'B', 'C', 'E', 'A', 'object']
F.__mro__:        ['F', 'D', 'B', 'C', 'E', 'A', 'object']


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [4]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "base_secret"
        self._semi_private = "base_semi"
        self.public = "base_public"

class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        self.__secret_value = "child_secret"
        self._semi_private = "child_semi"

    def dump_dict(self):
        return self.__dict__

base = SecureBase()
child = SecureChild()

print("SecureBase.__dict__:", base.__dict__)
print("SecureChild.__dict__:", child.__dict__)
print()
print("dir(base):", [x for x in dir(base) if 'secret' in x or 'private' in x])
print("dir(child):", [x for x in dir(child) if 'secret' in x or 'private' in x])

print("\nAccess mangled from outside:")
print("base._SecureBase__secret_value:", base._SecureBase__secret_value)
print("child._SecureBase__secret_value:", child._SecureBase__secret_value)
print("child._SecureChild__secret_value:", child._SecureChild__secret_value)

SecureBase.__dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'base_semi', 'public': 'base_public'}
SecureChild.__dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'child_semi', 'public': 'base_public', '_SecureChild__secret_value': 'child_secret'}

dir(base): ['_SecureBase__secret_value', '_semi_private']
dir(child): ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private']

Access mangled from outside:
base._SecureBase__secret_value: base_secret
child._SecureBase__secret_value: base_secret
child._SecureChild__secret_value: child_secret


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [5]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")
    def __init__(self, x):
        self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

print("NoSlots:", describe_instance(n))
print("WithSlots:", describe_instance(w))
print("WithSlotsWeak:", describe_instance(ww))

print("\nDynamic attribute addition:")
try:
    n.z = 3
    print("NoSlots: OK")
except AttributeError as e:
    print(f"NoSlots: {e}")

try:
    w.z = 3
    print("WithSlots: OK")
except AttributeError as e:
    print(f"WithSlots: {e}")

print("\nWeakref creation:")
try:
    wr_n = weakref.ref(n)
    print(f"NoSlots: OK, ref={wr_n()}")
except TypeError as e:
    print(f"NoSlots: {e}")

try:
    wr_w = weakref.ref(w)
    print(f"WithSlots: OK, ref={wr_w()}")
except TypeError as e:
    print(f"WithSlots: {e}")

try:
    wr_ww = weakref.ref(ww)
    print(f"WithSlotsWeak: OK, ref={wr_ww()}")
except TypeError as e:
    print(f"WithSlotsWeak: {e}")

NoSlots: {'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y']}
WithSlots: {'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
WithSlotsWeak: {'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}

Dynamic attribute addition:
NoSlots: OK
WithSlots: 'WithSlots' object has no attribute 'z' and no __dict__ for setting new attributes

Weakref creation:
NoSlots: OK, ref=<__main__.NoSlots object at 0x112208ec0>
WithSlots: cannot create weak reference to 'WithSlots' object
WithSlotsWeak: OK, ref=<__main__.WithSlotsWeak object at 0x112253f90>


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [6]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        if parent:
            if weak_parent:
                self.parent = weakref.ref(parent)
            else:
                self.parent = parent
        else:
            self.parent = None
        self.weak = weak_parent

    def get_parent(self):
        if self.parent is None:
            return None
        if self.weak:
            return self.parent()
        return self.parent

def refcount(obj):
    return sys.getrefcount(obj) - 1

a = Node("A")
b = Node("B", parent=a)
a.parent = b

print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
gc.collect()
print("After deletion: garbage collected")

c = Node("C")
d = Node("D", parent=c, weak_parent=True)

print("\nWeak cycle refcounts:", refcount(c), refcount(d))

wr_c = weakref.ref(c)
print("wr_c before:", wr_c())

del c
gc.collect()

print("wr_c after deletion:", wr_c())

Strong cycle refcounts: 2 2
After deletion: garbage collected

Weak cycle refcounts: 1 1
wr_c before: <__main__.Node object at 0x110c49950>
wr_c after deletion: None
